# 演習4. 壊さないように測る

ここまでは「一番遅い段は Infer です」と**答えを先に教えてもらって**進めてきました。
実際には**測らないと分かりません。** そして手を動かす前にやることは決まっています ――
**まず直列（1スレッド）で、データ1個の内訳を測る。**
スレッドが動いていると「正味」と「待っていた時間」が混ざるからです。

この演習の主題はこれです。

> **測ること自体が、測りたいものを変えてしまう。**

デバッグ表示を1行足しただけで1フレームが 12ms 遅くなることがあり、
しかも**各段の測定値は正しいまま**なので、**変わったことに気づけません。**

> **最初に下のセル（`cq.h`）を1回だけ実行してください。** 4-3 で使います。

In [ ]:
%%writefile cq.h
// ============================================================
//  cq.h ―― この演習で使うスレッドセーフなキュー
//  中身は読まなくてよい。使うのは push / pop / size の3つだけ。
// ============================================================
#pragma once
#include <queue>
#include <mutex>
#include <condition_variable>

template <typename T>
class ConcurrentQueue {
public:
    explicit ConcurrentQueue(std::size_t capacity) : capacity_(capacity) {}

    // 入れる。満杯なら空くまで待つ
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v);
        if (q_.size() > peak_) peak_ = q_.size();
        lk.unlock();
        can_pop_.notify_one();
    }

    // 取り出す。空なら来るまで待つ（失敗しない）
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop();
        lk.unlock();
        can_push_.notify_one();
        return v;
    }

    std::size_t size() const { std::lock_guard<std::mutex> g(mtx_); return q_.size(); }
    std::size_t peak() const { std::lock_guard<std::mutex> g(mtx_); return peak_; }   // 観測用：並んだ最大数

private:
    std::queue<T> q_;
    std::size_t capacity_;
    std::size_t peak_ = 0;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_;    // 「取り出せるようになった」
    std::condition_variable can_push_;   // 「入れられるようになった」
};

## 4-1. 【予測クイズ】計測は、いくらかかるのか

計測でやることは2種類だけです。**時計を読むこと**と、**結果を出すこと**。

```cpp
auto t = steady_clock::now();          // 時計を読む
busy += 経過時間;                       // 足し込む
cout << "time= " << ... << flush;      // 結果を出す
```

**実行前に予測してください。** 1回あたり何ナノ秒くらいでしょうか。**どれが一番高い**でしょうか。

（1000万回まわすので、表示は `> /dev/null` に捨てます。測定結果は `cerr` に出るので画面に残ります）

In [ ]:
%%writefile ex04a.cpp
#include <iostream>
#include <iomanip>
#include <sstream>
#include <chrono>
using namespace std::chrono;

volatile long long sink = 0;

template <class F>
double ns_per_call(int n, F f) {          // 1回あたりのナノ秒
    auto t0 = steady_clock::now();
    for (int i = 0; i < n; i++) f(i);
    return duration_cast<nanoseconds>(steady_clock::now() - t0).count() / (double)n;
}

int main() {
    long long acc = 0;

    // (1) 時計を1回読む
    double c_clock = ns_per_call(1000000, [&](int) {
        sink += steady_clock::now().time_since_epoch().count() & 1;
    });

    // (2) 測って足し込む（時計2回 + 引き算 + 足し算）＝ 計測1回ぶん
    double c_lap = ns_per_call(1000000, [&](int) {
        auto a = steady_clock::now();
        auto b = steady_clock::now();
        acc += duration_cast<nanoseconds>(b - a).count();
    });

    // (3) 1行ぶんの文字列を作るだけ（画面には出さない）
    double c_str = ns_per_call(100000, [&](int i) {
        std::ostringstream os;
        os << "\nrunYOLO preprocessing time= " << i << " [mS]\n";
        sink += os.str().size();
    });

    // (4) cout に1行流す（flush なし）
    double c_out = ns_per_call(20000, [&](int i) {
        std::cout << "\nrunYOLO preprocessing time= " << i << " [mS]\n";
    });

    // (5) cout に1行流して flush する
    double c_flush = ns_per_call(20000, [&](int i) {
        std::cout << "\nrunYOLO preprocessing time= " << i << " [mS]\n" << std::flush;
    });

    std::cerr << std::fixed << std::setprecision(0);
    std::cerr << "\n1回あたりの値段（このマシン、この出力先で）\n\n";
    std::cerr << std::setw(10) << c_clock << " ns   steady_clock::now() を1回読む\n";
    std::cerr << std::setw(10) << c_lap   << " ns   時計2回 + 引き算 + 足し込み（計測1回ぶん）\n";
    std::cerr << std::setw(10) << c_str   << " ns   1行ぶんの文字列を作る（表示はしない）\n";
    std::cerr << std::setw(10) << c_out   << " ns   cout に1行流す（flush なし）\n";
    std::cerr << std::setw(10) << c_flush << " ns   cout に1行流して flush する\n";
    std::cerr << "\n(" << acc << " " << sink << ")\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 ex04a.cpp -o ex04a && ./ex04a > /dev/null

### 結果 ―― 時計は安い。表示が高い

```
        23 ns   steady_clock::now() を1回読む
        47 ns   時計2回 + 引き算 + 足し込み（計測1回ぶん）
       266 ns   1行ぶんの文字列を作る（表示はしない）
        99 ns   cout に1行流す（flush なし）
       419 ns   cout に1行流して flush する
```

**数字そのものはマシンと出力先で変わります。** 変わらないのは**並び順**です ――
**時計 ≪ 文字列を作る ≦ 表示する**。

**時計を読むのは 23ns。** 1フレームに14回読んでも 0.3µs、1フレーム 10ms なら **0.003%**。
気にする必要はありません。一方、**文字列を1行作るだけで 266ns** ―― 時計10回分以上で、
しかもまだ画面に出す前です。

> **測るのは安い。出すのが高い。**

### しかも「高さ」は出力先で3桁変わる

上の値は出力先が `/dev/null` だからです。実際には：

```
ファイル / dev/null       0.2 〜 0.4 us
パイプ（| tee など）       0.8 us
SSH 端末                  数百 us
シリアルコンソール          約 3 ms      ← 1行で
```

シリアルの値は計算で出ます。**115200 baud = 毎秒 11520 バイト**なので 1バイト 87µs、
36バイトの行で **3.1ms**。段ごとに出せば**1フレームに4行 = 12.4ms**。
1フレーム 82ms の処理なら 15%、**13ms の処理なら処理時間と同じかそれ以上**です。

> **同じ `cout` が、Colab では無害で、シリアルコンソールでは処理より重い。**

### いちばん怖いのは「高いのに、数字に出ない」こと

`cout` を**計測区間の外**に置いてあったとします。これ自体は正しい書き方です。

```cpp
auto t0 = now();
...前処理...
auto t1 = now();
cout << "pre time= " << (t1-t0) << flush;   // 計測区間の外
auto t2 = now();                            // dpu の計測はここから
```

**`pre` も `dpu` も `post` も、正しい値を出し続けます。**
それなのに全体は 12ms/フレーム遅い。気づく方法は1つだけです。

> **「各段の合計」と「1フレームの実測時間」を並べて出し、差を見る。**

この差を以降 **「計測外」** と呼びます。ここが大きければ、**測り漏らしている仕事があります。**

## 4-2. 【予測クイズ】直列で1フレームの内訳を測る

### 壊さないように測る ―― 5つの作法

- **① 実行中は足し込むだけ。表示は全部終わってから1回だけ**
- **② 各スレッドが自分専用の記録場所を持つ**（共有しないので鍵が要らない ⇒ 直列化も起きない）
- **③ `steady_clock` と `microseconds`**
  （`system_clock` は時刻同期で飛ぶ。`milliseconds` は切り捨てなので `2.9ms` が `2` になる）
- **④ 立ち上がりの数フレームを捨てる**（メモリ確保やキャッシュの空振りを含むため）
- **⑤ 平均と最大の両方を出し、「合計 = 実測」を検算する**

### 道具の形

この5つを満たす道具は、驚くほど小さく書けます。

```cpp
void lap(int id, TP& t) {                   // t から今までを段 id に足す
    TP now = steady_clock::now();
    long long us = duration_cast<microseconds>(now - t).count();
    t = now;                                // 物差しを進める
    if (warm_) return;                      // ④ 立ち上がりは捨てる
    st_[id].sum += us;                      // ① 足し込むだけ
    if (us > st_[id].max) st_[id].max = us; // ⑤ 最大も
    frame_ += us;                           // ⑤ 検算用の合計
}
```

**`t` を参照で受け取って進めていく**のが要点です。段と段のあいだに隙間ができないので、
**測り漏らしがそのまま「計測外」に出ます。**

### 測る対象

仕事の中身は、**推論の軽いモデルを使った画像処理を模したもの**です。

```
              CPU     外部
read          13ms      -     デコード
pre            9ms      -     前処理
dpu             -      2ms    アクセラレータが処理している
post            2ms     -     後処理
show             -     30ms   画面が描いている（10個に1個は 60ms）
```

> **「CPU」は CPU 自身が計算している区間**、**「外部」はアクセラレータや画面が仕事をしている区間**です。
> 外部の区間はコアを手放しているので、**その裏では他のスレッドがコアを使えます。**
> CPU の区間はコアを占有するので、そうはいきません。**この2つは分けて測ります。**

**まだスレッドは組みません。** 直列のまま、1フレームの内訳を出すだけです。

**実行前に予測してください。** 一番大きい段はどれで、それは「CPU」でしょうか「外部」でしょうか。
**スレッドを組めば速くなる**でしょうか。

In [ ]:
%%writefile ex04c.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <chrono>
#include <atomic>
using namespace std::chrono;

using TP = steady_clock::time_point;
static inline TP now_() { return steady_clock::now(); }

// ============ 計測の道具 ============ 実行中は足し込むだけ。表示は最後に1回だけ
class Profiler {
public:
    int stage(const char* name, bool ext) {         // ext = 外部（コアを手放している）段
        st_.push_back({name, ext, 0, 0});
        return (int)st_.size() - 1;
    }
    void lap(int id, TP& t) {                       // t から今までを段 id に足し、t を進める
        TP n = now_();
        long long us = duration_cast<microseconds>(n - t).count();
        t = n;
        if (warm_) return;                          // 立ち上がりは捨てる
        st_[id].sum += us;
        if (us > st_[id].max) st_[id].max = us;
        frame_ += us;                               // 検算用の合計
    }
    void frame_end(TP fstart) {
        long long total = duration_cast<microseconds>(now_() - fstart).count();
        if (warm_) { if (++skipped_ >= WARMUP) { warm_ = false; t0_ = now_(); } return; }
        other_ += (total > frame_ ? total - frame_ : 0);   // 測り漏らし ＝ 計測外
        frame_ = 0;
        frames_++;
    }
    void report() const {
        double sec = duration_cast<microseconds>(now_() - t0_).count() / 1e6;
        std::cout << std::fixed << std::setprecision(1)
                  << frames_ << " 個ぶん（先頭 " << WARMUP << " 個は捨てた）   "
                  << (frames_ / sec) << " FPS\n\n"
                  << "    平均     最大   種類  段\n";
        for (const auto& s : st_)
            std::cout << std::setw(8) << (s.sum / 1000.0 / frames_) << "ms"
                      << std::setw(7) << (s.max / 1000.0) << "ms"
                      << (s.ext ? "   外部  " : "   CPU   ") << s.name << "\n";
        std::cout << std::setw(8) << (other_ / 1000.0 / frames_) << "ms"
                  << std::setw(9) << "-" << "         計測外\n";
    }
private:
    struct S { const char* name; bool ext; long long sum, max; };
    static const int WARMUP = 3;
    std::vector<S> st_;
    long long frame_ = 0, other_ = 0;
    int frames_ = 0, skipped_ = 0;
    bool warm_ = true;
    TP t0_ = now_();
};

// ============ 仕事の中身（推論の軽いモデルを使った画像処理を模したもの） ============
long calib = 0;
std::atomic<long> sink{0};        // 最適化で消されないための置き場（発展課題4-1 では複数スレッドから足す）
long burn(long n) { long s = 0; for (long i = 0; i < n; i++) s += (i * 2654435761u) % 7; return s; }
void cpu_ms(int ms)  { sink.fetch_add(burn(calib * ms), std::memory_order_relaxed); }   // CPU が計算する
void ext_ms(int ms)  { std::this_thread::sleep_for(milliseconds(ms)); }                      // 外部にまかせている

void calibrate() {
    long n = 100000;
    for (;;) {
        TP t = now_(); sink.fetch_add(burn(n), std::memory_order_relaxed);
        auto us = duration_cast<microseconds>(now_() - t).count();
        if (us > 30000) { calib = n * 1000 / us; break; }
        n *= 2;
    }
}

int main() {
    calibrate();
    Profiler p;
    int READ = p.stage("read (デコード)", false);
    int PRE  = p.stage("pre  (前処理)",   false);
    int DPU  = p.stage("dpu  (推論)",     true);      // 投げたあとはアクセラレータの仕事
    int POST = p.stage("post (後処理)",   false);
    int SHOW = p.stage("show (表示)",     true);

    for (int i = 0; i < 33; i++) {
        TP t = now_(), fstart = t;
        cpu_ms(13);                       p.lap(READ, t);
        cpu_ms(9);                        p.lap(PRE,  t);
        ext_ms(2);                        p.lap(DPU,  t);
        cpu_ms(2);                        p.lap(POST, t);
        ext_ms(i % 10 == 0 ? 60 : 30);    p.lap(SHOW, t);
        p.frame_end(fstart);
    }
    p.report();
    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 ex04c.cpp -o ex04c && ./ex04c

### 結果の読み方

```
30 個ぶん（先頭 3 個は捨てた）   16.4 FPS

    平均     最大   種類  段
    13.9ms   20.5ms   CPU   read (デコード)
     9.6ms   14.3ms   CPU   pre  (前処理)
     2.1ms    2.2ms   外部  dpu  (推論)
     2.2ms    3.4ms   CPU   post (後処理)
    33.2ms   60.2ms   外部  show (表示)
     0.0ms        -         計測外
```

読む順番は3つです。

**1つめ ―― まず「計測外」を見る。** `0.0ms` なら測り漏らしなし。
ここが大きければ、**他のどの数字も信用してはいけません。** まずそれを潰します。

**2つめ ―― 一番大きい段と、その「種類」（CPU か外部）を見る。**
ここでは `show` の 33.2ms（外部）。**スレッドを1本も足さないまま犯人が分かりました。**

**3つめ ―― 「最大」の列を見る。** `show` は平均 33ms なのに**最大 60ms**。
**平均だけでは、この振れに気づけません。** 振れる段は、パイプラインにしたとき
キューを詰まらせます（演習3-1【B】）。

## 4-3. 【予測クイズ】組んでから測る

同じ仕事を、今度は**3段パイプライン**にします。中身は 4-2 とまったく同じです。

```
Read  = read 13ms（CPU）
Infer = pre 9ms（CPU） + dpu 2ms（外部） + post 2ms（CPU） = 13ms
Show  = show 30ms（外部、10個に1個は 60ms）
```

### 組むと、4-2 の測り方が使えなくなる

直列なら「**各段を足せば1フレーム**」でした。パイプラインでは3段が重なるので成り立ちません。
足せば 13 + 13 + 33 = 59ms ですが、実際の1フレームはもっと短くなります。
**「計測外」で検算する手も使えません。**

代わりに、**段ごとに3つに分けて**測ります。

```
取り出し待ち   q.pop() で、上流からデータが来るのを待っていた時間
正味           自分の仕事をしていた時間
入れ待ち       q.push() で、下流が受け取ってくれるのを待っていた時間
```

記録場所は**スレッドごとに別**（作法②）なので、鍵は要りません。

**実行前に予測してください。** FPS はいくつになるでしょうか。
そして、**3つの列がどうなっている段がボトルネック**でしょうか。

In [ ]:
%%writefile ex04d.cpp
// 4-3  同じ仕事を3段パイプラインに組んで測る（4-2 は直列だった）
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <chrono>
#include <atomic>
#include "cq.h"
using namespace std::chrono;
using TP = steady_clock::time_point;

// ---- 仕事の中身。4-2 とまったく同じ ----
long calib = 0;
std::atomic<long> sink{0};        // 3本から足すので atomic（volatile では守れない）
long burn(long n) { long s = 0; for (long i = 0; i < n; i++) s += (i * 2654435761u) % 7; return s; }
void cpu_ms(int ms) { sink.fetch_add(burn(calib * ms), std::memory_order_relaxed); }   // CPU が計算する
void ext_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }                 // 外部にまかせる
void calibrate() {
    long n = 100000;
    for (;;) {
        TP t = steady_clock::now(); sink.fetch_add(burn(n), std::memory_order_relaxed);
        auto us = duration_cast<microseconds>(steady_clock::now() - t).count();
        if (us > 30000) { calib = n * 1000 / us; return; }
        n *= 2;
    }
}

const int N = 33, WARMUP = 3;      // 立ち上がりの3個は捨てる（作法④）
const std::size_t CAP = 4;

// 段ごとの記録。各スレッドが自分専用に持つので、鍵は要らない（作法②）
struct Stat { long long pop_us = 0, work_us = 0, push_us = 0; };

// t から今までの時間(us)を返し、t を進める（作法③）
long long lap(TP& t) {
    TP n = steady_clock::now();
    long long us = duration_cast<microseconds>(n - t).count();
    t = n;
    return us;
}

int main() {
    calibrate();
    ConcurrentQueue<int> q1(CAP), q2(CAP);      // Read→Infer、Infer→Show
    Stat sr, si, ss;
    TP tstart;

    std::thread rd([&] {
        for (int i = 0; i < N; i++) {
            TP t = steady_clock::now();
            cpu_ms(13);                  sr.work_us += (i < WARMUP ? 0 : lap(t));   // read
            q1.push(i);                  sr.push_us += (i < WARMUP ? 0 : lap(t));   // 入れ待ち
        }
    });
    std::thread in([&] {
        for (int i = 0; i < N; i++) {
            TP t = steady_clock::now();
            int v = q1.pop();            si.pop_us  += (i < WARMUP ? 0 : lap(t));   // 取り出し待ち
            cpu_ms(9); ext_ms(2); cpu_ms(2);
                                         si.work_us += (i < WARMUP ? 0 : lap(t));   // pre+dpu+post
            q2.push(v);                  si.push_us += (i < WARMUP ? 0 : lap(t));   // 入れ待ち
        }
    });
    std::thread sh([&] {
        for (int i = 0; i < N; i++) {
            TP t = steady_clock::now();
            q2.pop();                    ss.pop_us  += (i < WARMUP ? 0 : lap(t));   // 取り出し待ち
            if (i == WARMUP) tstart = steady_clock::now();
            ext_ms(i % 10 == 0 ? 60 : 30);
                                         ss.work_us += (i < WARMUP ? 0 : lap(t));   // show
        }
    });
    rd.join(); in.join(); sh.join();

    // ---- 表示は全部終わってから1回だけ（作法①）----
    double sec = duration_cast<microseconds>(steady_clock::now() - tstart).count() / 1e6;
    int n = N - WARMUP;
    std::cout << std::fixed << std::setprecision(1)
              << "3段パイプライン（Read / Infer / Show を1人ずつ、容量 " << CAP << "）  実測 "
              << (n / sec) << " FPS\n\n"
              << "  取り出し待ち     正味     入れ待ち   段\n";
    auto line = [&](const char* nm, const Stat& s) {
        std::cout << std::setw(11) << (s.pop_us / 1000.0 / n) << "ms"
                  << std::setw(9)  << (s.work_us / 1000.0 / n) << "ms"
                  << std::setw(11) << (s.push_us / 1000.0 / n) << "ms   " << nm << "\n";
    };
    line("Read",  sr);
    line("Infer", si);
    line("Show",  ss);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread -O2 ex04d.cpp -o ex04d && ./ex04d

### 結果 ―― 待っていない段が、犯人

```
3段パイプライン（Read / Infer / Show を1人ずつ、容量 4）  実測 30.2 FPS

  取り出し待ち     正味     入れ待ち   段
        0.0ms     15.2ms       10.5ms   Read
        0.1ms     15.4ms       14.7ms   Infer
        0.0ms     33.1ms        0.0ms   Show     ← どこも待っていない
```

**Show だけが、どこでも待っていません。**

- Read と Infer は `push` で待っています ―― 下流の Show が受け取らないので、キューが満杯のまま
- Show は `pop` で待っていません ―― キューにはいつもデータが並んでいる

> **待っていない段がボトルネックです。** 他の段は、その段のせいで待たされています。

**FPS も一致します。** Show の正味 33.1ms から `1000 / 33.1 ≒ 30 FPS`。
4-2 の表から計算できた上限A（演習3-2）と同じ値です。
**直列で測った内訳だけで、組む前に FPS を当てられた**ことになります。

（`正味` の値は Colab の混み具合で上下します。**見るべきは3つの列の関係**です）

### 直列と、組んだあと

```
             直列（4-2）                   3段パイプライン（4-3）
測り方        各段を足す                    段ごとに 取り出し待ち / 正味 / 入れ待ち
検算          合計 = 1フレーム              使えない（段が重なる）
犯人の見つけ方  一番大きい段                  待っていない段
FPS          16.4                          30.2
```

**どちらでも犯人は同じ Show** でした。組んでから測ると待ち時間が混ざるので、
先に直列で測るほうが早いのです。

## 4-4. 測り方の落とし穴

（4-2 の**作法①〜⑤**は「どう書くか」の話でした。こちらは「どう使うか」の話です）

- **落とし穴1　1回だけ測って判断する。** 同じプログラムでも実行のたびに10%は動きます。
  **数回まわして最小値と中央値を見る。1回目は捨てる**
- **落とし穴2　途中で打ち切る。** 打ち切ると、キューに残っている仕事を数え忘れます
- **落とし穴3　「速くなった」の中身を言えない。** FPS が上がったのか、レイテンシが縮んだのか。
  演習3で見たとおり、**この2つは同時には良くなりません**
- **落とし穴4　いきなり細かく測る。** 全体の FPS → 直列で1個ぶんの内訳 → 犯人の段の中をさらに分ける。
  **はじめの2段階で答えが出ることがほとんど**です
- **落とし穴5　条件を変えたのに、前の数字と比べる。** コンパイルオプション、入力、表示の有無 ――
  どれか1つでも変えたら、比較できるのは同じ条件で測り直した数字だけです

## 演習の地図 ―― ここまでで扱ったこと

```
演習1  スレッドとパイプライン   thread / join、待つ仕事と計算する仕事、レイテンシとスループット
演習2  スレッドセーフなキュー   守らない共有は壊れる、push / pop / size、容量
演習3  パイプラインを組む       容量の決め方、バックプレッシャ、2つの上限
演習4  壊さないように測る       測る値段、計測外、5つの作法、直列で測ってから組んで確かめる
```